# Description

In this notebook, we benchmark SINDy algorithm on the Korns benchmarks.

In [ ]:
from config.korns_config import BENCH, SINDY, FEATURE_NAMES
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, SRFitResult

from dataclasses import dataclass
from typing import List, Dict, Tuple, Callable
import numpy as np
import sympy as sp

from pysindy.feature_library import PolynomialLibrary, CustomLibrary, ConcatLibrary
from pysindy.optimizers import STLSQ


def to_sympy(feature_name: str, symbols: Dict[str, sp.Symbol]) -> sp.Expr:
    return sp.sympify(feature_name.replace("^", "**").replace(" ", "*"), locals=symbols)


def make_custom_library(cfg) -> CustomLibrary:
    clip_exp = float(cfg.clip_exp)
    log_eps = float(cfg.log_eps)
    div_eps = float(cfg.div_eps)
    sqrt_abs = bool(cfg.sqrt_abs)

    def sin_(x):  return np.nan_to_num(np.sin(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def cos_(x):  return np.nan_to_num(np.cos(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def tan_(x):  return np.nan_to_num(np.tan(x),  nan=0.0, posinf=0.0, neginf=0.0)
    def tanh_(x): return np.nan_to_num(np.tanh(x), nan=0.0, posinf=0.0, neginf=0.0)

    def exp_(x):
        y = np.exp(np.clip(x, -clip_exp, clip_exp))
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def log_(x):
        y = np.log(np.abs(x) + log_eps)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def sqrt_(x):
        y = np.sqrt(np.abs(x)) if sqrt_abs else np.sqrt(x)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    def div_(a, b):
        y = a / (b + div_eps)
        return np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    unary: Dict[str, Tuple[Callable, Callable]] = {
        "sin":  (sin_,  lambda x: f"sin({x})"),
        "cos":  (cos_,  lambda x: f"cos({x})"),
        "tan":  (tan_,  lambda x: f"tan({x})"),
        "tanh": (tanh_, lambda x: f"tanh({x})"),
        "exp":  (exp_,  lambda x: f"exp({x})"),
        "log":  (log_,  lambda x: f"log({x})"),
        "sqrt": (sqrt_, lambda x: f"sqrt({x})"),
    }

    funcs: List[Callable] = []
    names: List[Callable] = []

    for op in cfg.unary_ops:
        f, n = unary[op]
        funcs.append(f)
        names.append(n)

    if "/" in cfg.binary_ops:
        funcs.append(div_)
        names.append(lambda a, b: f"({a})/({b})")

    return CustomLibrary(library_functions=funcs, function_names=names, include_bias=False)


@dataclass
class SINDyKornsRegressor:
    name: str = "sindy"
    cfg: object = SINDY

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        X_train = np.asarray(X_train, dtype=np.float64)
        X_test = np.asarray(X_test, dtype=np.float64)
        y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)

        input_names = list(FEATURE_NAMES[: X_train.shape[1]])

        poly = PolynomialLibrary(
            degree=int(self.cfg.poly_degree),
            include_interaction=bool(self.cfg.include_interaction),
            include_bias=bool(self.cfg.include_bias),
        )
        custom = make_custom_library(self.cfg)
        library = ConcatLibrary([poly, custom])

        library.fit(X_train)

        theta_train = np.asarray(library.transform(X_train), dtype=np.float64)
        theta_test = np.asarray(library.transform(X_test), dtype=np.float64)

        theta_train = np.nan_to_num(theta_train, nan=0.0, posinf=0.0, neginf=0.0)
        theta_test = np.nan_to_num(theta_test, nan=0.0, posinf=0.0, neginf=0.0)

        m = float(self.cfg.max_feature_abs)
        theta_train = np.clip(theta_train, -m, m)
        theta_test = np.clip(theta_test, -m, m)

        feature_names = library.get_feature_names(input_features=input_names)

        k = int(getattr(self.cfg, "max_library_features", 0) or 0)
        if k > 0 and theta_train.shape[1] > k:
            theta_train = theta_train[:, :k]
            theta_test = theta_test[:, :k]
            feature_names = feature_names[:k]

        if bool(self.cfg.drop_nonfinite_rows):
            keep = np.isfinite(theta_train).all(axis=1) & np.isfinite(y_train).all(axis=1)
            theta_train = theta_train[keep]
            y_train = y_train[keep]

        optimizer = STLSQ(
            threshold=float(self.cfg.threshold),
            alpha=float(self.cfg.alpha),
            max_iter=int(self.cfg.max_iter),
            normalize_columns=bool(self.cfg.normalize_columns),
            verbose=bool(self.cfg.stlsq_verbose),
        )
        optimizer.fit(theta_train, y_train)

        coef = np.asarray(optimizer.coef_, dtype=np.float64)
        coef = coef[0] if coef.ndim == 2 else coef
        intercept = float(np.atleast_1d(getattr(optimizer, "intercept_", 0.0))[0])

        coef = np.nan_to_num(coef, nan=0.0, posinf=0.0, neginf=0.0)
        intercept = float(np.nan_to_num(intercept, nan=0.0, posinf=0.0, neginf=0.0))

        y_pred_train = (theta_train @ coef.reshape(-1, 1)).reshape(-1) + intercept
        y_pred_test = (theta_test @ coef.reshape(-1, 1)).reshape(-1) + intercept

        expr = None
        try:
            symbols = {n: sp.Symbol(n) for n in input_names}
            tol = float(self.cfg.coef_zero_tol)

            terms = []
            for c, fname in zip(coef.tolist(), feature_names):
                if abs(c) <= tol:
                    continue
                terms.append(sp.Float(c) * to_sympy(fname, symbols))

            expr = sp.Add(*terms) if terms else sp.Float(0.0)
            if abs(intercept) > tol:
                expr = expr + sp.Float(intercept)
        except Exception:
            expr = None

        return SRFitResult(
            expr=expr,
            y_pred_train=np.asarray(y_pred_train, dtype=np.float64).reshape(-1),
            y_pred_test=np.asarray(y_pred_test, dtype=np.float64).reshape(-1),
            metadata=None,
        )


run_cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(run_cfg.hdf5_path)

rows = run_benchmark(
    datasets=datasets,
    algorithms=[SINDyKornsRegressor()],
    config=run_cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=SINDY.results_csv_path,
)


[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] sindy
[RUN START] run_id=0 seed=14176308741000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 4.1005e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
         1 ... 3.2854e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
[PRED] 24.3*x3 + 1.56999999999999
[RUN START] run_id=1 seed=14176309741000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 4.1005e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
         1 ... 3.2854e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
[PRED] 24.3*x3 + 1.56999999999999
[RUN START] run_id=2 seed=14176310741000
 Iteration ... |y - Xw|^2 ...  a * |w|_2 ...      |w|_0 ... Total error: |y - Xw|^2 + a * |w|_2
         0 ... 4.1005e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
         1 ... 3.2854e-06 ... 3.2756e+00 ...          2 ... 3.2756e+00
[PRED] 24.3*x3 + 1.56999999999999
[RUN START] run